<a href="https://colab.research.google.com/github/ANDRESCHAVERRA/002_EstudiantesAprendizajeEstadistico/blob/main/NOTEBOOK_DE_LIMPIEZA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from scipy import stats

In [3]:
# 1. Carga de datos
urls = [
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vQnfcp_x9Aq8wyg9CGjMFdLaQsXauV6yktGhQLi5Q7oj0wHaLJaDepS4v3BLukBhax3-uzpDoS74SN7/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vSDWCjjvFk965oXs9_S1PhqbwSwcfu7DEeJnkDrlzG0cBXQFNh0N28ILY8jFpwC1xOFZqvGxmIld6iJ/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vQ0E0ev9SDgl-m3KSc0EjoyFzcJsFxC_e2H60ckiFXtqG1ZmBvSxZye_JuysbO0Rl2cbYvN29fxWASn/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vRp_SXHlb3RG4DevqbZchmXGWgcA3noRS-TQEfn5atnx1JrD2c90a2LES7oW1r5ckH8NpStGOuBRmFN/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vQRP1U_kswi3_YqYAMAwYIGhWIzWuPnxQO3fhz2jNAKmqu0HzPcInWAD82VwVrh0fGL0oDSBmTGEjsB/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vT-cFmh21DhDjfym6olsEMcc_plbp5s2Gzy1OWECYXvT2Mk5WEk4VdSoy-gpgd3lWJ7EVXKmXuACMXz/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vR_bcRo3riHK36EyQc5TED3nJIXdPrluEbe6n_8sOfcWdpVj67Ujmzf0rDmOmrcrTLrOboEdC6svori/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vRq84ulxNsMF5-6QMbaH9SsDG4jDgWjB1flQt0veQyMNkWP64XugItikeAAEkN5-ne5WKgdywF2nu2O/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vRyu1JN0WS6F4DXAO4OfDJBMNiuEyyKI_3TdpL6QlVccTdAUfQbQDAzg0lgL2yfxFDkhJw6yp2S9S7U/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vSfRoKmke06FwPGqdIk9Wd1EQGf7k5V5DHWvfKqMO3AaLrLAnF7_wW8kaQPjjihNbj0D0vaDt7TBuet/pub?output=csv',
    'https://docs.google.com/spreadsheets/d/e/2PACX-1vQThcoo2C5LaS9qDJTNgYnGAD8vDDR9gtKqVj7D4zfQXGT7p2C_wePKZEFPSzVsgZzHt3OQ7uhtW_dW/pub?output=csv'
]

In [4]:
import pandas as pd
import hashlib

def generar_id_estudiante(nombre):
    """
    Genera un ID único y consistente basado en el nombre.
    Usa un hash truncado para que sea siempre el mismo para el mismo nombre.
    """
    # Limpiamos el nombre para evitar variaciones por espacios o mayúsculas
    nombre_limpio = str(nombre).strip().upper()
    # Creamos un hash SHA-256 y tomamos los primeros 8 caracteres
    return hashlib.sha256(nombre_limpio.encode()).hexdigest()[:8]

def procesar_simulacros_con_id(urls):
    """
    Concatena los datos y aplica una anonimización reversible mediante IDs hash.
    """
    list_df = []
    for u in urls:
        try:
            tdf = pd.read_csv(u)
            tdf.columns = tdf.columns.str.strip().str.replace('\n', ' ')
            list_df.append(tdf)
        except Exception as e:
            print(f"Error en URL {u}: {e}")

    if not list_df: return pd.DataFrame(), {}

    df_raw = pd.concat(list_df, ignore_index=True)

    # Limpieza básica
    df_clean = df_raw[['Nombre del estudiante', 'Puntos obtenidos']].copy()
    df_clean['Puntos obtenidos'] = pd.to_numeric(df_clean['Puntos obtenidos'], errors='coerce')
    df_clean = df_clean.dropna()

    # =========================================================
    # GENERACIÓN DE ID CONSISTENTE Y REVERSIBLE
    # =========================================================
    # 1. Creamos el ID basado en el nombre (siempre dará el mismo resultado)
    df_clean['ID_Estudiante'] = df_clean['Nombre del estudiante'].apply(generar_id_estudiante)

    # 2. Creamos el "Diccionario Maestro" (La Llave)
    # Esto te permite revertir el proceso: ID -> Nombre
    llave_reversion = df_clean[['ID_Estudiante', 'Nombre del estudiante']].drop_duplicates()
    llave_reversion = llave_reversion.set_index('ID_Estudiante')['Nombre del estudiante'].to_dict()
    # =========================================================

    # 3. Estructuración de la tabla para el modelo
    df_clean['n_simulacro'] = df_clean.groupby('ID_Estudiante').cumcount() + 1
    df_clean['n_simulacro'] = 'SIMULACRO ' + df_clean['n_simulacro'].astype(str)

    df_final = df_clean.pivot(index='ID_Estudiante',
                              columns='n_simulacro',
                              values='Puntos obtenidos')

    df_final['TOTAL PRESENTADOS'] = df_final.notna().sum(axis=1)

    columnas_sim = sorted([c for c in df_final.columns if 'SIMULACRO' in c],
                           key=lambda x: int(x.split(' ')[1]))

    df_final = df_final[['TOTAL PRESENTADOS'] + columnas_sim]

    return df_final, llave_reversion

procesar_simulacros_con_id(urls)[0]

n_simulacro,TOTAL PRESENTADOS,SIMULACRO 1,SIMULACRO 2,SIMULACRO 3,SIMULACRO 4,SIMULACRO 5,SIMULACRO 6,SIMULACRO 7,SIMULACRO 8,SIMULACRO 9,SIMULACRO 10,SIMULACRO 11
ID_Estudiante,,,,,,,,,,,,
0922fd01,11,49.0,34.0,38.0,56.0,50.0,61.0,39.0,29.0,34.0,40.0,57.0
44eae406,2,42.0,35.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6d713c9e,6,35.0,54.0,37.0,20.0,17.0,38.0,NaN,NaN,NaN,NaN,NaN
797cd4a5,11,43.0,20.0,32.0,44.0,38.0,56.0,39.0,32.0,32.0,32.0,37.0
7b8a0848,8,23.0,21.0,42.0,41.0,33.0,28.0,25.0,27.0,NaN,NaN,NaN
8106ec62,10,29.0,30.0,33.0,40.0,33.0,43.0,31.0,36.0,25.0,39.0,NaN
87ff928a,2,31.0,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9eb05fd9,4,28.0,44.0,29.0,46.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
d00323ec,10,39.0,33.0,37.0,45.0,55.0,36.0,41.0,24.0,43.0,54.0,NaN


In [5]:
procesar_simulacros_con_id(urls)[1]

{'de059642': 'Yina Paola Argote Ortega',
 '7b8a0848': 'Julio González Arango',
 '9eb05fd9': 'Paulina Zapata Álvarez',
 'ee7433df': 'Heylly Andrea Londoño Ramírez',
 'd00323ec': 'Daniel Felipe Villamizar Castellanos',
 '0922fd01': 'Carolina Moncada Mesa',
 '6d713c9e': 'Juan Pablo Villada David',
 '797cd4a5': 'Felipe Pareja León',
 '8106ec62': 'Karen Andrea Madrid Álvarez',
 'e79dfa2e': 'Emanuel Guiral García',
 '44eae406': 'Natally Gómez Álvarez',
 '87ff928a': 'Ana Sofía Mejía López'}

In [11]:
# Guardar en csv
DF_final = procesar_simulacros_con_id(urls)[0]
DF_final.to_csv('AVANZADO 2025-1.csv', index=True)